# Run the volatility study for `patch_s2` (once)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shmorsi/tsfm-from-scratch/blob/main/notebooks/vol_study_patch_s2.ipynb)

`PREREGISTRATION_VOL.md` (tag `prereg-vol-v1`) names four models for the realized-volatility
study. The two baselines, a random walk and HAR-RV, were scored before the deadline. The
other two were not (`DEVIATIONS.md`): the headline model `patch_s2` zero-shot, and the same
weights after fine-tuning. Its checkpoint stayed on Google Drive, so they cannot be run
from a clone. This notebook runs both, once, on the series and with the study code that
scored the baselines.

**Time, before Run all.** No duration is stored for this study, so none is promised here.
The work is small next to pretraining: 4,886 forecast origins of 22 days for each
run, and at most 300 fine-tuning steps on short windows. The run also installs the pinned
environment and, with `VERIFY_RAW` on, downloads the Binance archives again. A free Colab
runtime can disconnect: each finished run is written to the Drive `WORK` folder, and Run all
again continues from there. A finished run is not run a second time.

**Before Run all**

1. Merge the commit you want to run, or set `COMMIT` below to that branch. The notebook
   clones `COMMIT` (default `main`).
2. Runtime -> Change runtime type -> T4 GPU.
3. Leave `CHECKPOINT_PATH` when the run folder is already at the default Drive path. It
   has to be that `patch_s2` run, not a neighbour such as `bins_s2`. The weights that are
   used are the ones whose SHA-256 was recorded with the validation run:
   `85c46256cefa665b7eb0822f9877831563df85e22345670b40c4a115c16d0fd2`.

**What the pre-registration fixes, and this notebook follows**

* Data: daily realized volatility of BTCUSDT and ETHUSDT. The two series are the files in
  `results/finance/`, and they must hash to the values in `results/finance/rvol_summary.json`,
  the files the baselines were scored on. With `VERIFY_RAW` on, the Binance archives are
  also downloaded again, checked against Binance's published checksums and
  `locks/binance.lock.json`, and must rebuild those two files byte for byte.
* Test period: every day from 2020-01-01 is a forecast origin, 22 days ahead.
* Model 3: the checkpoint chosen in `PREREGISTRATION.md`, seed 0, zero-shot.
* Model 4: the same weights fine-tuned on data up to 2019-06-30 with the pretraining
  objective (AdamW, learning rate 1e-4, batch 64, at most 300 steps). The step with the
  lowest weighted quantile loss on the validation period (2019-07-01 to 2019-12-31,
  origins every 5 days) is kept.
* Scores: pooled weighted quantile loss over both symbols and all test origins, as a ratio
  to the random walk with a 95% moving-block bootstrap interval (blocks of 22 origins).
* Primary comparisons: fine-tuned against zero-shot, and each against HAR-RV. One model is
  better than another only if the whole interval of its loss ratio lies below 1.
* Scored once. The script stops if the checked-out commit already holds either run, and
  `scripts/harvest_colab_output.py` refuses to replace a committed run with different rows.

**Where the code is more specific than the protocol.** Nothing is changed for this run.
These are stated so that the write-up can state them too.

* The model reads at most the last 2,048 days before an origin, the inference setting of
  `PREREGISTRATION.md` section 5. The protocol says the context is all data up to the
  origin. From 2023 the history is longer than 2,048 days.
* During fine-tuning the validation loss is measured every 25 steps, so the kept step is
  the best of steps 0, 25, 50 and so on. If step 0 is the best, the fine-tuned model is
  the zero-shot model and the two runs give the same rows.
* AdamW runs with betas 0.9 and 0.95 and no weight decay. The protocol names only AdamW.

**What this run is not.** It is one checkpoint, seed 0. It measures statistical forecast
scores only. No trading or economic-value claim of any kind is made from it.

**After it finishes.** The last cell prints where the harvest file is. Download
`harvest_vol_patch_s2.txt` from the Drive `WORK` folder (a copied notebook print can
truncate, and a truncated file fails the SHA-256 check). Then, on a local clone of the
same commit:

    uv run python scripts/harvest_colab_output.py harvest_vol_patch_s2.txt
    uv run tsfm docs build

Commit the files the harvest script wrote and the docs that command regenerated, and add a
dated entry to `DEVIATIONS.md`, whose 2026-10-15 table says the model was not run on this
study. Do not hand-edit numbers. The harvest file holds:

* `results/finance/summary.json` (rebuilt by `tsfm vol report`; this replaces the previous file)
* `results/finance/tsfm_zero_shot/` and `results/finance/tsfm_finetuned/`: `run.json` and
  `per_origin.csv` each. The rows travel packed against the random-walk rows already in
  the repository, and the harvest script restores the exact files.

The same folder gets `release/tsfm_patch_s2_seed0_model.pt`, a copy of the weights that
were scored, with their SHA-256 beside them. Attach that file to a GitHub release if the
weights should be public.


In [ ]:
# Parameters
# ============================================================================
# CHECKPOINT_PATH: the patch_s2 run folder on Drive. ckpt.pt (training checkpoint) or
# model.pt (inference weights) both work: the script checks the run config and the
# parameter count, then uses the file whose SHA-256 matches the validation run.
CHECKPOINT_PATH = "/content/drive/MyDrive/tsfm/runs/patch_s2/ckpt.pt"
# VERIFY_RAW: download the Binance archives again and require them to rebuild the
# committed series exactly. Set to False only if that download fails; the harvest file
# then records that the re-check was skipped.
VERIFY_RAW = True
# ============================================================================
REPO = "shmorsi/tsfm-from-scratch"
COMMIT = "main"  # or a full commit SHA, or a branch name
WORK = "/content/drive/MyDrive/tsfm/vol_study"  # finished runs, harvest file, weights copy

In [ ]:
# 1. Mount Drive (checkpoint in, finished runs and the harvest file out)
import os

from google.colab import drive

drive.mount("/content/drive")
assert os.path.isfile(CHECKPOINT_PATH), f"CHECKPOINT_PATH not found: {CHECKPOINT_PATH}"
os.makedirs(WORK, exist_ok=True)

In [ ]:
# 2. Clone the repo at COMMIT, with tags (the script checks for prereg-vol-v1)
import os
import subprocess

if not os.path.exists("/content/tsfm"):
    subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", "/content/tsfm"], check=True)
subprocess.run(["git", "-C", "/content/tsfm", "fetch", "-q", "--force", "--tags", "origin"], check=True)
remote = f"origin/{COMMIT}"
known = subprocess.run(["git", "-C", "/content/tsfm", "rev-parse", "-q", "--verify", remote], capture_output=True)
ref = remote if known.returncode == 0 else COMMIT
subprocess.run(["git", "-C", "/content/tsfm", "checkout", "-q", "--force", "--detach", ref], check=True)
print(subprocess.run(["git", "-C", "/content/tsfm", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
tagged = subprocess.run(["git", "-C", "/content/tsfm", "merge-base", "--is-ancestor", "prereg-vol-v1", "HEAD"])
assert tagged.returncode == 0, "prereg-vol-v1 is not in this commit's history; nothing is run"


In [ ]:
# 3. Install (uv, locked dependencies, CUDA 12.6 PyTorch build)
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
import os

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["TSFM_HOME"] = "/content/tsfm_home"
%cd /content/tsfm
!uv sync --locked --extra cu126 -q


In [ ]:
# 4. GPU check with the PyTorch this repo pins (not Colab's preinstalled build)
import subprocess

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout)
probe = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        "import torch; name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''; print(torch.__version__); print(torch.cuda.is_available()); print(name)",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(probe.stdout)
lines = probe.stdout.splitlines()
assert len(lines) >= 3, "GPU probe did not print version, availability and device name"
assert lines[1] == "True", "No GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# 5. Check the checkpoint, the tag, the inputs and that nothing was run before
!uv run python scripts/run_vol_study.py --checkpoint "{CHECKPOINT_PATH}" --work "{WORK}" --dry-run

In [ ]:
# 6. Run once: input re-check, zero-shot, fine-tuned, tsfm vol report, harvest file,
#    weights copy, tsfm docs build. Re-run after a disconnect.
RAW = "" if VERIFY_RAW else "--skip-raw-check"
!uv run python scripts/run_vol_study.py --checkpoint "{CHECKPOINT_PATH}" --work "{WORK}" --device cuda {RAW}

In [ ]:
# 7. What to download. Take WORK/harvest_vol_patch_s2.txt from Drive (do not copy a
#    truncated print). On a local clone of this COMMIT:
#    uv run python scripts/harvest_colab_output.py harvest_vol_patch_s2.txt
#    uv run tsfm docs build
import os

harvest = f"{WORK}/harvest_vol_patch_s2.txt"
print("download this file from Drive:", harvest, f"({os.path.getsize(harvest):,} bytes)")
for name in sorted(os.listdir(f"{WORK}/parts")):
    size = os.path.getsize(os.path.join(WORK, "parts", name))
    print("  the same blocks in parts:", f"{WORK}/parts/{name}", f"({size:,} bytes)")
with open(harvest) as f:
    for line in f:
        if line.startswith("==="):
            break
        print(line, end="")
print("weights that were scored, for a GitHub release:")
print(open(f"{WORK}/release/tsfm_patch_s2_seed0_model.pt.sha256").read())
!git -C /content/tsfm status --short